In [1]:
import os
import torch 
from d2l import torch as d2l

ModuleNotFoundError: No module named 'd2l'

In [2]:
## Download and preprocess the dataset

In [3]:
class MTFraEng(d2l.DataModule):
    def _download(self):
        d2l.extract(d2l.download(d2l.DATA_URL+'fra-eng.zip',self.root,"94646ad1522d915e7b0f9296181140edcf86a4f5"))

        with open(self.root + '/fra-eng/fra.txt',encoding='utf-8') as f:
            return f.read()

data = MTFaEng()
raw_text = data._download()
print(raw_text[:75])

NameError: name 'd2l' is not defined

In [4]:
## d2l

In [5]:
@d2l.add_to_class(MTFraEng)  #@save
def _preprocess(self, text):
    # Replace non-breaking space with space
    text = text.replace('\u202f', ' ').replace('\xa0', ' ')
    # Insert space between words and punctuation marks
    no_space = lambda char, prev_char: char in ',.!?' and prev_char != ' '
    out = [' ' + char if i > 0 and no_space(char, text[i - 1]) else char
           for i, char in enumerate(text.lower())]
    return ''.join(out)

text = data._preprocess(raw_text)
print(text[:80])

NameError: name 'd2l' is not defined

## Tokenizations

In [6]:
@d2l.add_to_class(MTFraEng)
def _tokenize(self,text,max_examples=None):
    src,tgt = [] , []
    for i , line in enumerate(text.split('\n')):
        if max_examples and i > max_examples: break
        parts = line.split("\t")
        if len(parts) == 2:
            # Skip empty tokens
            src.append([t for t in f'{parts[0]} <eos>'.split(' ') if t])
            tgt.append([t for t in f'{parts[1]} <eos>'.split(' ') if t])
    return src,tgt

src , tgt = data._tokenize(text)
src[:6],tgt[:6]

NameError: name 'd2l' is not defined

## Let’s plot the histogram of the number of tokens per text sequence. In this simple English–French dataset, most of the text sequences have fewer than 20 tokens.



In [ ]:
#@save
def show_list_len_pair_hist(legend, xlabel, ylabel, xlist, ylist):
    """Plot the histogram for list length pairs."""
    d2l.set_figsize()
    _, _, patches = d2l.plt.hist(
        [[len(l) for l in xlist], [len(l) for l in ylist]])
    d2l.plt.xlabel(xlabel)
    d2l.plt.ylabel(ylabel)
    for patch in patches[1].patches:
        patch.set_hatch('/')
    d2l.plt.legend(legend)

show_list_len_pair_hist(['source', 'target'], '# tokens per sequence',
                        'count', src, tgt);

In [7]:
## Loading seq of firth length

In [8]:
@d2l.add_to_class(MTFraEng)
def __init__(self,batch_size,num_steps=9,num_trains=512,num_val=128):
    super(MTFraEng,self).__init__()
    self.save_hyperparameters()
    self.arrays,self.src_vocab,self.tgt_vocab = self._build_arrays(
        self._download()
    )

@d2l.add_to_class(MTFraEng)  #@save
def _build_arrays(self, raw_text, src_vocab=None, tgt_vocab=None):
    def _build_array(sentences, vocab, is_tgt=False):
        pad_or_trim = lambda seq, t: (
            seq[:t] if len(seq) > t else seq + ['<pad>'] * (t - len(seq)))
        sentences = [pad_or_trim(s, self.num_steps) for s in sentences]
        if is_tgt:
            sentences = [['<bos>'] + s for s in sentences]
        if vocab is None:
            vocab = d2l.Vocab(sentences, min_freq=2)
        array = torch.tensor([vocab[s] for s in sentences])
        valid_len = (array != vocab['<pad>']).type(torch.int32).sum(1)
        return array, vocab, valid_len
    src, tgt = self._tokenize(self._preprocess(raw_text),
                              self.num_train + self.num_val)
    src_array, src_vocab, src_valid_len = _build_array(src, src_vocab)
    tgt_array, tgt_vocab, _ = _build_array(tgt, tgt_vocab, True)
    return ((src_array, tgt_array[:,:-1], src_valid_len, tgt_array[:,1:]),
            src_vocab, tgt_vocab)
        

NameError: name 'd2l' is not defined

In [9]:
### Reading the dataset

@d2l.add_to_class(MTFraEng)
def get_dataloader(self,train):
    idx = slice(0,self.num_train) if train else slice(self.num_train,None)

    return self.get_tensorloader(self.arrays,train,idx)


NameError: name 'd2l' is not defined

In [10]:
## Loading the MT Fra eng 

In [11]:
data = MTFraEng(Batch_size=3)
src,tgt= src_valid_len , label = next(iter(data.train_dataloader()))
print('source:', src.type(torch.int32))
print('decoder input:', tgt.type(torch.int32))
print('source len excluding pad:', src_valid_len.type(torch.int32))
print('label:', label.type(torch.int32))

NameError: name 'MTFraEng' is not defined